# GOBAI-O2 — how deep is the oxygen niche in the Bering Sea?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nmfs-opensci/gobai-rfrom-icechunks/blob/main/GOBAI-O2/gobai-o2-oxygen-niche-demo.ipynb)

A short demo of **GOBAI HR** (NOAA PMEL), gridded dissolved oxygen mapped from Argo floats and
ship profiles: weekly, 0.25°, 58 pressure levels to 1975 dbar, 1993–2025. It is published as
a virtual Icechunk store of about 830 GB that opens in about a second. We map the **depth of
the oxygen niche** in the Bering Sea and see how it has changed over 33 years.

No credentials needed. Runs on Google Colab (Python 3.12).

In [ ]:
%pip install -q "icechunk>=2.2" "xarray>=2026.7" "zarr>=3.4" "dask[array]" matplotlib

## Open the dataset

The store holds only metadata; the values are read by byte range from the netCDF files in the
public NOAA bucket `gs://noaa-oar-gobai`. `chunks=None` keeps it lazy without building a huge
dask graph — select first, then chunk. A zarr warning about the `numcodecs` shuffle/zlib
codecs is expected and harmless.

In [ ]:
import icechunk as ic, xarray as xr, zarr, numpy as np
import matplotlib.pyplot as plt, matplotlib.patches as mpatches

zarr.config.set({"async.concurrency": 64})

SRC = "gs://noaa-oar-gobai/netcdf/v202606/"
repo = ic.Repository.open(
    ic.gcs_storage(bucket="noaa-oar-gobai", prefix="icechunk/v202606", anonymous=True),
    authorize_virtual_chunk_access=ic.containers_credentials({SRC: ic.gcs_credentials(anonymous=True)}),
)
ds = xr.open_zarr(repo.readonly_session("main").store, consolidated=False, chunks=None)
ds

## The oxygen niche

Fish need oxygen. Below about **60 µmol kg⁻¹** (roughly 2 mg L⁻¹, a common hypoxia
threshold) most can't stay long. So one simple measure of the oxygen niche is its **depth**:
the pressure of the first level where O₂ drops below 60. Everything above it is habitable
water; a shallower value means a thinner niche.

```python
hypoxic_depth = o2.mean_pressure.where(o2 < 60).min("mean_pressure")
```

It is NaN where the water column never drops below 60, and on land and on the shallow shelf,
where GOBAI has no data (it is built from profiles in deep water).

In [ ]:
THRESHOLD = 60   # µmol/kg

def hypoxic_depth(o2):
    """Pressure (dbar) of the first level where O2 < THRESHOLD."""
    return o2.mean_pressure.where(o2 < THRESHOLD).min("mean_pressure")

## Map: Bering Sea, one week

Longitude in this store runs 0–360, so the Bering Sea (160°E–155°W) is one plain selection.

In [ ]:
o2 = ds.o2.sel(time="2010-07-16", method="nearest").sel(latitude=slice(50, 67), longitude=slice(160, 205)).load()
depth_map = hypoxic_depth(o2)

# The box used for the time series below: the deep Aleutian Basin.
box = dict(latitude=(54, 58), longitude=(180, 188))

fig, ax = plt.subplots(figsize=(9, 6))
depth_map.plot(ax=ax, cmap="viridis_r", vmin=100, vmax=700,
               cbar_kwargs={"label": f"depth where O₂ first < {THRESHOLD} µmol/kg (dbar)"})
ax.add_patch(mpatches.Rectangle((box["longitude"][0], box["latitude"][0]),
                                box["longitude"][1] - box["longitude"][0],
                                box["latitude"][1] - box["latitude"][0],
                                fill=False, ec="magenta", lw=2))
ax.set(title=f"GOBAI-O2 oxygen niche depth, week of {str(o2.time.values)[:10]} (magenta: time-series box)",
       xlabel="longitude (°E, 0–360)", ylabel="latitude")
ax.set_facecolor("0.8")   # land, shelf with no data, or never below the threshold
plt.show()

## Time series: is the niche getting thinner?

The box-mean niche depth over the Aleutian Basin for **every week, 1993–2025**. Only levels
down to 1000 dbar are needed. The store is chunked 100 weeks at a time, so a long series over
a small box is the cheap case here: this reads the whole record in about a minute.

In [ ]:
b = ds.o2.sel(latitude=slice(*box["latitude"]), longitude=slice(*box["longitude"]),
             mean_pressure=slice(0, 1000))
b = b.chunk({"time": 100, "mean_pressure": 1})     # match the store's chunks
weights = np.cos(np.deg2rad(b.latitude))            # grid cells shrink toward the pole

series = hypoxic_depth(b).weighted(weights).mean(["latitude", "longitude"]).compute()
annual = series.groupby("time.year").mean()
annual.to_series().round(0)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(series.time, series, color="tab:blue", lw=0.6, alpha=0.4, label="weekly")
ax.plot([np.datetime64(f"{y}-07-01") for y in annual.year.values], annual, "o-",
        color="tab:blue", label="annual mean")
ax.invert_yaxis()   # deeper is lower
ax.set(ylabel="niche depth (dbar)",
       title=f"Depth where O₂ first < {THRESHOLD} µmol/kg, Aleutian Basin "
             f"{box['latitude'][0]}–{box['latitude'][1]}°N, {box['longitude'][0]}–{box['longitude'][1]}°E")
ax.legend()
plt.show()

In the 1990s the oxygenated layer here reached about 400–460 dbar; since 2016 it has
reached only about 310–350. The habitable layer has thinned by roughly 70 m, most of it in a
step around 2016, squeezing the oxygen niche toward the surface.

GOBAI is a gridded product mapped from floats and ship profiles, not a direct measurement at
each point, and the files are labelled *preliminary*; profile coverage is thinner in the early
years.

**More:** [GOBAI-O2/README.md](https://github.com/nmfs-opensci/gobai-rfrom-icechunks/blob/main/GOBAI-O2/README.md)
covers the store, the netCDFs, reading from R, and the browser viewer.